In [ ]:
# ----------------------------
# 1. IMPORT LIBRARIES
# ----------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ----------------------------
# 2. GENERATE SYNTHETIC WEATHER DATA
# ----------------------------
np.random.seed(42)
days = 2000
dates = pd.date_range(start='2018-01-01', periods=days, freq='D')

# Seasonal temperature (sinusoidal with noise)
temp = 20 + 15 * np.sin(2 * np.pi * np.arange(days) / 365) + np.random.normal(0, 3, days)

# Humidity: anti-correlated with temperature
humidity = 60 - 0.4 * temp + np.random.normal(0, 5, days)
humidity = np.clip(humidity, 20, 100)  # Keep in valid range

# Wind speed: random but bounded
wind_speed = np.abs(np.random.normal(15, 5, days))
wind_speed = np.clip(wind_speed, 5, 40)

# Create DataFrame
df = pd.DataFrame({
    'Date': dates,
    'Temperature': temp,
    'Humidity': humidity,
    'Wind Speed': wind_speed
})

# Display first 10 rows
print("✅ First 10 rows:")
print(df.head(10))

# Check missing values
print("\n✅ Missing values:", df.isnull().sum().sum())

# Plot temperature trend
plt.figure(figsize=(12, 4))
plt.plot(df['Date'], df['Temperature'], color='orange')
plt.title('Daily Temperature Over Time')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# 3. PREPROCESSING
# ----------------------------
# Select features
features = ['Temperature', 'Humidity', 'Wind Speed']
data = df[features].values

# Normalize
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Create sequences (past 14 days → next day temperature)
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length, 0])  # temperature is index 0
    return np.array(X), np.array(y)

SEQ_LENGTH = 14
X, y = create_sequences(data_scaled, SEQ_LENGTH)

# Split: 70% train, 15% val, 15% test
split1 = int(0.7 * len(X))
split2 = int(0.85 * len(X))

X_train, y_train = X[:split1], y[:split1]
X_val, y_val = X[split1:split2], y[split1:split2]
X_test, y_test = X[split2:], y[split2:]

print(f"✅ Data shapes:")
print(f"   Train: X={X_train.shape}, y={y_train.shape}")
print(f"   Val:   X={X_val.shape}, y={y_val.shape}")
print(f"   Test:  X={X_test.shape}, y={y_test.shape}")

In [ ]:
# ----------------------------
# 4. BUILD SIMPLE RNN MODEL
# ----------------------------
model = Sequential([
    SimpleRNN(64, input_shape=(SEQ_LENGTH, len(features)), return_sequences=False),
    Dropout(0.2),
    Dense(1, activation='linear')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print(model.summary())

In [ ]:
# ----------------------------
# 5. TRAIN MODEL
# ----------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    verbose=1
)

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Model MAE')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# 6. EVALUATE ON TEST SET
# ----------------------------
y_pred_scaled = model.predict(X_test)

# Inverse transform to original scale (only Temperature column)
# Create dummy array to invert scaling
dummy = np.zeros((len(y_pred_scaled), len(features)))
dummy[:, 0] = y_pred_scaled.flatten()
y_pred = scaler.inverse_transform(dummy)[:, 0]

dummy2 = np.zeros((len(y_test), len(features)))
dummy2[:, 0] = y_test
y_test_actual = scaler.inverse_transform(dummy2)[:, 0]

# Metrics
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
mae = mean_absolute_error(y_test_actual, y_pred)
r2 = r2_score(y_test_actual, y_pred)

print(f"✅ Test Set Performance:")
print(f"   RMSE: {rmse:.2f} °C")
print(f"   MAE:  {mae:.2f} °C")
print(f"   R²:   {r2:.4f}")

# Plot predictions vs actual
plt.figure(figsize=(12, 5))
plt.plot(y_test_actual[:100], label='Actual', color='blue')
plt.plot(y_pred[:100], label='Predicted', color='red', linestyle='--')
plt.title('Predicted vs Actual Temperature (First 100 Test Days)')
plt.xlabel('Day')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# 7. FORECAST NEXT 7 DAYS
# ----------------------------
# Start with last sequence from test set
last_sequence = X_test[-1].copy()  # shape: (14, 3)
forecast_scaled = []

for _ in range(7):
    # Predict next temperature (scaled)
    next_temp_scaled = model.predict(last_sequence[np.newaxis, :], verbose=0)[0, 0]
    forecast_scaled.append(next_temp_scaled)
    
    # Update sequence: drop oldest, append new prediction (with placeholder for other features)
    new_row = last_sequence[-1].copy()
    new_row[0] = next_temp_scaled  # update temperature
    # Keep humidity/wind as last known (simple assumption)
    last_sequence = np.vstack([last_sequence[1:], new_row])

# Inverse transform forecast
dummy = np.zeros((7, len(features)))
dummy[:, 0] = forecast_scaled
forecast = scaler.inverse_transform(dummy)[:, 0]

# Plot forecast vs last 30 days of actual data
last_30_actual = y_test_actual[-30:]

plt.figure(figsize=(12, 5))
plt.plot(range(30), last_30_actual, label='Actual (Last 30 Days)', marker='o')
plt.plot(range(30, 37), forecast, label='Forecast (Next 7 Days)', marker='o', color='red')
plt.axvline(x=29, color='gray', linestyle='--', label='Forecast Start')
plt.title('7-Day Temperature Forecast')
plt.xlabel('Day (Relative)')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.show()

print("✅ 7-Day Forecast (°C):")
for i, temp in enumerate(forecast, 1):
    print(f"   Day {i}: {temp:.2f} °C")